In [101]:
# Setup for Kaggle or Google Colab environments
# Run this cell to install any missing dependencies!
!pip install -q pandas numpy scikit-learn scipy matplotlib seaborn mplsoccer tqdm

# Player Style Clustering — Master Pipeline

A condensed, end-to-end run of the full pipeline: extract (or load) player-season features → preprocess → K-Means (RobustScaler, k=5) → validate (position purity/NMI + KNN stability). This notebook applies the configuration already justified in `01_data_and_eda.ipynb`–`03_evaluation.ipynb` directly, without re-deriving it — see those notebooks for the EDA, scaler/k selection experiments, and cluster interpretation behind these choices. Running all cells regenerates every pipeline output under `data/`.

## Setup

In [102]:
%matplotlib inline
import json
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import normalized_mutual_info_score

# Read from the local repo's data/ if present; otherwise (e.g. a bare Kaggle/Colab notebook)
# fall back to a writable working directory - /kaggle/input/... is read-only, so outputs
# can never be written there.
_LOCAL_DATA_DIR = next((p for p in [Path("../data"), Path("data"), Path("../../data")] if p.exists()), None)
if _LOCAL_DATA_DIR is not None:
    DATA_DIR = _LOCAL_DATA_DIR.resolve()
else:
    DATA_DIR = Path("/kaggle/working/data")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Using data directory:", DATA_DIR)

ID_COLS = ["player_id", "player_name", "primary_team", "competitions_played",
           "n_matches_played", "total_minutes_played"]
FEATURE_COLS = [
    "shots_p90", "np_goals_p90", "avg_shot_distance", "dribbles_p90", "dribble_success_rate",
    "touches_p90", "touches_in_box_p90", "passes_p90", "pass_completion_pct", "key_passes_p90",
    "assists_p90", "crosses_p90", "through_balls_p90", "long_balls_p90", "avg_pass_length",
    "pressures_p90", "tackles_p90", "tackle_success_rate", "interceptions_p90",
    "interception_success_rate", "ball_recoveries_p90", "clearances_p90", "aerial_won_p90",
    "fouls_committed_p90", "fouls_won_p90", "avg_location_x", "avg_location_y",
    "std_location_x", "std_location_y",
]
HIDDEN_COLS = ["primary_position"]

# Finalized choices from 02_modeling.ipynb / 03_evaluation.ipynb - not re-derived here
FINAL_SCALER = "RobustScaler"
FINAL_K = 5
FINAL_N_NEIGHBORS = 15

Using data directory: /kaggle/working/data


## Step 1 — Extract player-season features from StatsBomb events

Skipped if raw StatsBomb JSON isn't found locally (`data/raw/` is gitignored) — falls back to the pre-extracted CSV already committed at `data/extract-feature/`. See `01_data_and_eda.ipynb` for the full walkthrough of this logic.

In [103]:
COMPETITIONS = [(2, 27, "Premier League"), (11, 27, "La Liga"), (12, 27, "Serie A")]
MIN_MINUTES = 900
RAW_CANDIDATES = [DATA_DIR / "raw", Path("/kaggle/input/datasets/saurabhshahane/statsbomb-football-data/data")]
RAW_DIR = next((p for p in RAW_CANDIDATES if p.exists() and any(p.rglob("*.json"))), None)
EXTRACT_OUTPUT_CSV = DATA_DIR / "extract-feature" / "player_style_features_pl_laliga_seriea_1516.csv"

RED_CARD_NAMES = {"Red Card", "Second Yellow"}
TOUCH_EVENT_TYPES = {"Pass", "Shot", "Dribble", "Carry", "Ball Receipt*", "Miscontrol",
                      "Clearance", "Interception", "Ball Recovery", "Goal Keeper"}
TACKLE_SUCCESS_OUTCOMES = INTERCEPTION_SUCCESS_OUTCOMES = {"Won", "Success", "Success In Play", "Success Out"}


def discover_paths(base_dir):
    competitions_path, matches_index, events_index = None, {}, {}
    for p in base_dir.rglob("*.json"):
        parts = p.parts
        if p.name == "competitions.json" and competitions_path is None:
            competitions_path = p
        elif len(parts) >= 3 and parts[-3].lower() == "matches" and parts[-2].isdigit() and p.stem.isdigit():
            matches_index[(int(parts[-2]), int(p.stem))] = p
        elif len(parts) >= 2 and parts[-2].lower() == "events" and p.stem.isdigit():
            events_index[p.stem] = p
    return competitions_path, matches_index, events_index


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def compute_match_minutes(events):
    half_end_minutes = {}
    for e in events:
        if e["type"]["name"] == "Half End":
            half_end_minutes[e["period"]] = max(half_end_minutes.get(e["period"], 0), e["minute"])
    valid_periods = [p for p in half_end_minutes if p in (1, 2, 3, 4)]
    match_end_minute = max(half_end_minutes[p] for p in valid_periods) if valid_periods else 90

    start_minute, exit_minute, player_name, player_team = {}, {}, {}, {}

    def register_exit(pid, minute):
        if pid not in exit_minute or minute < exit_minute[pid]:
            exit_minute[pid] = minute

    for e in events:
        etype = e["type"]["name"]
        if etype == "Starting XI":
            team = e["team"]["name"]
            for p in e.get("tactics", {}).get("lineup", []):
                pid = p["player"]["id"]
                start_minute[pid] = 0
                player_name[pid], player_team[pid] = p["player"]["name"], team
        elif etype == "Substitution":
            register_exit(e["player"]["id"], e["minute"])
            replacement = e.get("substitution", {}).get("replacement")
            if replacement:
                on_id = replacement["id"]
                start_minute[on_id] = e["minute"]
                player_name[on_id], player_team[on_id] = replacement["name"], e["team"]["name"]
        elif etype == "Foul Committed" and e.get("foul_committed", {}).get("card", {}).get("name") in RED_CARD_NAMES:
            register_exit(e["player"]["id"], e["minute"])
        elif etype == "Bad Behaviour" and e.get("bad_behaviour", {}).get("card", {}).get("name") in RED_CARD_NAMES:
            register_exit(e["player"]["id"], e["minute"])

    return {pid: max(0, exit_minute.get(pid, match_end_minute) - s) for pid, s in start_minute.items()}, player_name, player_team


def scan_aerial_won(event):
    return any(isinstance(v, dict) and v.get("aerial_won") is True for v in event.values())


def aggregate_match_events(events):
    stats_by_player = defaultdict(lambda: defaultdict(float))
    positions = defaultdict(Counter)

    for e in events:
        player = e.get("player")
        if not player:
            continue
        pid, etype, row = player["id"], e["type"]["name"], stats_by_player[player["id"]]

        pos = e.get("position", {}).get("name")
        if pos:
            positions[pid][pos] += 1

        loc = e.get("location")
        if etype in TOUCH_EVENT_TYPES and loc:
            x, y = loc
            row["touches"] += 1
            row["sum_x"] += x; row["sum_y"] += y; row["sum_x2"] += x * x; row["sum_y2"] += y * y
            if x >= 102 and 18 <= y <= 62:
                row["touches_in_box"] += 1
        if scan_aerial_won(e):
            row["aerial_won"] += 1

        if etype == "Shot":
            shot = e.get("shot", {})
            row["shots"] += 1
            if shot.get("outcome", {}).get("name") == "Goal" and shot.get("type", {}).get("name") != "Penalty":
                row["np_goals"] += 1
            if loc:
                dx, dy = 120 - loc[0], 40 - loc[1]
                row["sum_shot_distance"] += (dx ** 2 + dy ** 2) ** 0.5
                row["n_shots_with_distance"] += 1
        elif etype == "Dribble":
            row["dribbles"] += 1
            if e.get("dribble", {}).get("outcome", {}).get("name") == "Complete":
                row["dribbles_complete"] += 1
        elif etype == "Pass":
            passv = e.get("pass", {})
            row["passes"] += 1
            if "outcome" not in passv:
                row["passes_complete"] += 1
            if passv.get("shot_assist"):
                row["key_passes"] += 1
            if passv.get("goal_assist"):
                row["assists"] += 1
            if passv.get("cross"):
                row["crosses"] += 1
            if passv.get("technique", {}).get("name") == "Through Ball":
                row["through_balls"] += 1
            length = passv.get("length")
            if length is not None:
                row["sum_pass_length"] += length
                row["n_passes_with_length"] += 1
                if length > 30:
                    row["long_balls"] += 1
        elif etype == "Pressure":
            row["pressures"] += 1
        elif etype == "Duel" and e.get("duel", {}).get("type", {}).get("name") == "Tackle":
            row["tackles"] += 1
            if e.get("duel", {}).get("outcome", {}).get("name") in TACKLE_SUCCESS_OUTCOMES:
                row["tackles_won"] += 1
        elif etype == "Interception":
            row["interceptions"] += 1
            if e.get("interception", {}).get("outcome", {}).get("name") in INTERCEPTION_SUCCESS_OUTCOMES:
                row["interceptions_won"] += 1
        elif etype == "Ball Recovery":
            row["ball_recoveries"] += 1
        elif etype == "Clearance":
            row["clearances"] += 1
        elif etype == "Foul Committed":
            row["fouls_committed"] += 1
        elif etype == "Foul Won":
            row["fouls_won"] += 1

    return stats_by_player, positions

In [104]:
if RAW_DIR is not None:
    _, matches_index, events_index = discover_paths(RAW_DIR)

    def get_match_ids(competition_id, season_id):
        return [m["match_id"] for m in load_json(matches_index[(competition_id, season_id)])]

    def get_events(match_id):
        return load_json(events_index[str(match_id)])

    global_minutes = defaultdict(float)
    global_matches_played = defaultdict(int)
    global_stats = defaultdict(lambda: defaultdict(float))
    global_positions = defaultdict(Counter)
    global_names, global_teams, global_competitions = {}, defaultdict(Counter), defaultdict(set)

    for comp_id, season_id, label in COMPETITIONS:
        for match_id in tqdm(get_match_ids(comp_id, season_id), desc=label):
            events = get_events(match_id)
            minutes, names, teams = compute_match_minutes(events)
            match_stats, positions = aggregate_match_events(events)
            for pid, m in minutes.items():
                if m <= 0:
                    continue
                global_minutes[pid] += m
                global_matches_played[pid] += 1
                global_names[pid] = names[pid]
                global_teams[pid][teams[pid]] += 1
                global_competitions[pid].add(label)
            for pid, row in match_stats.items():
                for k, v in row.items():
                    global_stats[pid][k] += v
            for pid, counter in positions.items():
                global_positions[pid].update(counter)

    rows = []
    for pid, minutes in global_minutes.items():
        row = global_stats[pid]
        factor = 90.0 / minutes if minutes > 0 else np.nan

        def rate(key):
            return row.get(key, 0.0) * factor

        def ratio(numer_key, denom_key):
            denom = row.get(denom_key, 0.0)
            return row.get(numer_key, 0.0) / denom if denom > 0 else np.nan

        def avg(sum_key, count_key):
            cnt = row.get(count_key, 0.0)
            return row.get(sum_key, 0.0) / cnt if cnt > 0 else np.nan

        touches = row.get("touches", 0.0)
        avg_x = row["sum_x"] / touches if touches > 0 else np.nan
        avg_y = row["sum_y"] / touches if touches > 0 else np.nan
        std_x = np.sqrt(max(row["sum_x2"] / touches - avg_x ** 2, 0)) if touches > 1 else np.nan
        std_y = np.sqrt(max(row["sum_y2"] / touches - avg_y ** 2, 0)) if touches > 1 else np.nan

        rows.append({
            "player_id": pid, "player_name": global_names.get(pid),
            "primary_team": global_teams[pid].most_common(1)[0][0] if global_teams[pid] else None,
            "competitions_played": ", ".join(sorted(global_competitions[pid])),
            "n_matches_played": global_matches_played[pid], "total_minutes_played": minutes,
            "shots_p90": rate("shots"), "np_goals_p90": rate("np_goals"),
            "avg_shot_distance": avg("sum_shot_distance", "n_shots_with_distance"),
            "dribbles_p90": rate("dribbles"), "dribble_success_rate": ratio("dribbles_complete", "dribbles"),
            "touches_p90": rate("touches"), "touches_in_box_p90": rate("touches_in_box"),
            "passes_p90": rate("passes"), "pass_completion_pct": ratio("passes_complete", "passes"),
            "key_passes_p90": rate("key_passes"), "assists_p90": rate("assists"), "crosses_p90": rate("crosses"),
            "through_balls_p90": rate("through_balls"), "long_balls_p90": rate("long_balls"),
            "avg_pass_length": avg("sum_pass_length", "n_passes_with_length"), "pressures_p90": rate("pressures"),
            "tackles_p90": rate("tackles"), "tackle_success_rate": ratio("tackles_won", "tackles"),
            "interceptions_p90": rate("interceptions"),
            "interception_success_rate": ratio("interceptions_won", "interceptions"),
            "ball_recoveries_p90": rate("ball_recoveries"), "clearances_p90": rate("clearances"),
            "aerial_won_p90": rate("aerial_won"), "fouls_committed_p90": rate("fouls_committed"),
            "fouls_won_p90": rate("fouls_won"), "avg_location_x": avg_x, "avg_location_y": avg_y,
            "std_location_x": std_x, "std_location_y": std_y,
            "primary_position": global_positions[pid].most_common(1)[0][0] if global_positions[pid] else None,
        })

    player_df_filtered = pd.DataFrame(rows)
    player_df_filtered = player_df_filtered[player_df_filtered["total_minutes_played"] >= MIN_MINUTES].reset_index(drop=True)

    EXTRACT_OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    player_df_filtered.to_csv(EXTRACT_OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Extracted and saved {player_df_filtered.shape[0]} players to {EXTRACT_OUTPUT_CSV}")
else:
    print(f"Raw data not found locally - using the pre-extracted CSV at {EXTRACT_OUTPUT_CSV}")

assert EXTRACT_OUTPUT_CSV.exists(), f"{EXTRACT_OUTPUT_CSV} not found and raw data is unavailable to regenerate it."

Premier League:   0%|          | 0/380 [00:00<?, ?it/s]

La Liga:   0%|          | 0/380 [00:00<?, ?it/s]

Serie A:   0%|          | 0/380 [00:00<?, ?it/s]

Extracted and saved 1016 players to /kaggle/working/data/extract-feature/player_style_features_pl_laliga_seriea_1516.csv


## Step 2 — Preprocess: median-impute, then scale 3 ways

See `02_modeling.ipynb` for why median imputation and why all 3 scaling variants are compared rather than assumed.

In [105]:
df = pd.read_csv(EXTRACT_OUTPUT_CSV, encoding="utf-8-sig")

X_raw = df[FEATURE_COLS].copy()
for col in FEATURE_COLS:
    if X_raw[col].isna().any():
        X_raw[col] = X_raw[col].fillna(X_raw[col].median())

X_unscaled = X_raw.copy()
X_standard = pd.DataFrame(StandardScaler().fit_transform(X_raw), columns=FEATURE_COLS, index=X_raw.index)
X_robust = pd.DataFrame(RobustScaler().fit_transform(X_raw), columns=FEATURE_COLS, index=X_raw.index)
X_by_scaler = {"Unscaled": X_unscaled, "StandardScaler": X_standard, "RobustScaler": X_robust}

id_df = df[ID_COLS].copy()
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for name, fname in [("Unscaled", "player_X_unscaled.csv"), ("StandardScaler", "player_X_standard_scaled.csv"),
                     ("RobustScaler", "player_X_robust_scaled.csv")]:
    pd.concat([id_df.reset_index(drop=True), X_by_scaler[name].reset_index(drop=True)], axis=1) \
        .to_csv(PROCESSED_DIR / fname, index=False, encoding="utf-8-sig")

pd.concat([id_df.reset_index(drop=True), df[HIDDEN_COLS].reset_index(drop=True)], axis=1) \
    .to_csv(PROCESSED_DIR / "player_y_hidden.csv", index=False, encoding="utf-8-sig")

print(f"Preprocessed {len(df)} players -> saved 4 files to {PROCESSED_DIR}")

Preprocessed 1016 players -> saved 4 files to /kaggle/working/data/processed


## Step 3 — K-Means (RobustScaler, k=5)

See `02_modeling.ipynb` for the Elbow/Silhouette/Gap Statistic sweep and Adjusted Rand Index comparison behind this choice.

In [106]:
final_km = KMeans(n_clusters=FINAL_K, n_init=10, random_state=42)
final_labels = final_km.fit_predict(X_by_scaler[FINAL_SCALER].values)

CLUSTER_DIR = DATA_DIR / "cluster"
CLUSTER_DIR.mkdir(parents=True, exist_ok=True)

cluster_output = id_df.copy()
cluster_output["cluster_id"] = final_labels
for name in X_by_scaler:
    cluster_output[f"cluster_id_{name}"] = KMeans(n_clusters=FINAL_K, n_init=10, random_state=42) \
        .fit_predict(X_by_scaler[name].values)

cluster_output.to_csv(CLUSTER_DIR / "player_clusters_k5.csv", index=False, encoding="utf-8-sig")
print(f"Saved {CLUSTER_DIR / 'player_clusters_k5.csv'}")
print(cluster_output["cluster_id"].value_counts().sort_index())

Saved /kaggle/working/data/cluster/player_clusters_k5.csv
cluster_id
0    194
1    152
2     73
3    198
4    399
Name: count, dtype: int64


## Step 4 — Validate: position purity/NMI + KNN stability

See `03_evaluation.ipynb` for the full crosstab, mixed-cluster analysis, cluster naming, and the `n_neighbors` sweep behind `FINAL_N_NEIGHBORS`.

In [107]:
position_group_map = {
    "Goalkeeper": "Goalkeeper",
    "Center Back": "Defense", "Left Center Back": "Defense", "Right Center Back": "Defense",
    "Left Back": "Defense", "Right Back": "Defense",
    "Left Wing Back": "Defense", "Right Wing Back": "Defense",
    "Center Defensive Midfield": "Midfield", "Left Defensive Midfield": "Midfield", "Right Defensive Midfield": "Midfield",
    "Center Attacking Midfield": "Midfield", "Left Center Midfield": "Midfield", "Right Center Midfield": "Midfield",
    "Left Midfield": "Midfield", "Right Midfield": "Midfield", "Right Attacking Midfield": "Midfield",
    "Center Forward": "Attack", "Left Center Forward": "Attack", "Right Center Forward": "Attack",
    "Left Wing": "Attack", "Right Wing": "Attack",
}

eval_df = cluster_output.merge(df[["player_id", "primary_position"]], on="player_id")
eval_df["position_group"] = eval_df["primary_position"].map(position_group_map)

crosstab = pd.crosstab(eval_df["cluster_id"], eval_df["position_group"])
overall_purity = crosstab.max(axis=1).sum() / crosstab.sum().sum()
nmi = normalized_mutual_info_score(eval_df["cluster_id"], eval_df["position_group"])
print(f"Overall purity vs. position_group: {overall_purity:.3f}")
print(f"NMI vs. position_group: {nmi:.3f}")

X_knn = X_robust[FEATURE_COLS].values
y_knn = cluster_output["cluster_id"].values
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
knn_acc = cross_val_score(KNeighborsClassifier(n_neighbors=FINAL_N_NEIGHBORS), X_knn, y_knn, cv=cv, scoring="accuracy")
print(f"KNN (n_neighbors={FINAL_N_NEIGHBORS}) 5-fold CV accuracy predicting cluster_id: {knn_acc.mean():.3f} +/- {knn_acc.std():.3f}")

Overall purity vs. position_group: 0.697
NMI vs. position_group: 0.546
KNN (n_neighbors=15) 5-fold CV accuracy predicting cluster_id: 0.966 +/- 0.008


## Results

Running this notebook end-to-end regenerates:
- `data/extract-feature/player_style_features_pl_laliga_seriea_1516.csv`
- `data/processed/player_X_unscaled.csv`, `player_X_standard_scaled.csv`, `player_X_robust_scaled.csv`, `player_y_hidden.csv`
- `data/cluster/player_clusters_k5.csv`

with purity/NMI/KNN-accuracy metrics matching `03_evaluation.ipynb` (overall purity ≈0.697, KNN CV accuracy ≈0.966 at `n_neighbors=15`). For the EDA, method comparisons, and cluster interpretation behind every choice made here, see `01_data_and_eda.ipynb` through `03_evaluation.ipynb`.